[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/dynamic-breakpoints.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239526-lesson-4-dynamic-breakpoints)

# 动态断点

## 回顾

我们讨论了人机协同（human-in-the-loop）的动机：

(1) `Approval`（审批）—— 我们可以中断 Agent，将状态展示给用户，并允许用户批准某个操作

(2) `Debugging`（调试）—— 我们可以回退图的执行以重现或避免问题

(3) `Editing`（编辑）—— 你可以修改状态

我们介绍了断点作为在特定步骤停止图的一般方法，它支持 `Approval` 等用例。

我们还展示了如何编辑图状态并引入人工反馈。

## 目标

断点由开发者在图编译期间设置，作用于特定节点。

但是，有时让图**动态地自我中断**会更有用！

这是一种内部断点，可以通过 `NodeInterrupt` 来实现。

这有一些具体的好处：

(1) 你可以有条件地进行中断（在节点内部根据开发者定义的逻辑）。

(2) 你可以向用户传达中断的原因（通过向 `NodeInterrupt` 传入任何你需要的信息）。

让我们创建一个图，在其中根据输入的长度抛出 `NodeInterrupt`。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_deepseek langgraph_sdk

In [ ]:
from IPython.display import Image, display

from typing_extensions import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import NodeInterrupt
from langgraph.graph import START, END, StateGraph

class State(TypedDict):
    input: str

def step_1(state: State) -> State:
    print("---Step 1---")
    return state

def step_2(state: State) -> State:
    # 如果输入长度大于 5 个字符，则选择性地抛出 NodeInterrupt
    if len(state['input']) > 5:
        raise NodeInterrupt(f"Received input that is longer than 5 characters: {state['input']}")
    
    print("---Step 2---")
    return state

def step_3(state: State) -> State:
    print("---Step 3---")
    return state

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

# 设置内存
memory = MemorySaver()

# 使用内存编译图
graph = builder.compile(checkpointer=memory)

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

让我们使用一个长度超过 5 个字符的输入来运行图。

In [ ]:
initial_input = {"input": "hello world"}
thread_config = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread_config, stream_mode="values"):
    print(event)

此时如果我们检查图状态，可以看到下一个要执行的节点是 `step_2`。


In [ ]:
state = graph.get_state(thread_config)
print(state.next)

我们可以看到 `Interrupt` 被记录到了状态中。

In [ ]:
print(state.tasks)

我们可以尝试从断点恢复图的执行。

但这样做只会重新运行同一个节点！

除非状态被更改，否则我们将一直卡在这里。

In [ ]:
for event in graph.stream(None, thread_config, stream_mode="values"):
    print(event)

In [ ]:
state = graph.get_state(thread_config)
print(state.next)

现在，我们可以更新状态。

In [ ]:
graph.update_state(
    thread_config,
    {"input": "hi"},
)

In [ ]:
for event in graph.stream(None, thread_config, stream_mode="values"):
    print(event)

### 通过 LangGraph API 使用

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而非视频中展示的桌面应用。它现在名为 _LangSmith Studio_（而非 _LangGraph Studio_）。详细设置说明请参见课程开头的"环境搭建"指南。你可以在[此处](https://docs.langchain.com/langsmith/studio)找到 Studio 的说明，在[此处](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。  
要启动本地开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

我们通过 SDK 连接到它。

In [ ]:
from langgraph_sdk import get_client

# 这是本地开发服务器的 URL
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

In [ ]:
thread = await client.threads.create()
input_dict = {"input": "hello world"}

async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="dynamic_breakpoints",
    input=input_dict,
    stream_mode="values",):
    
    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])

In [ ]:
current_state['next']

In [ ]:
await client.threads.update_state(thread['thread_id'], {"input": "hi!"})

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="dynamic_breakpoints",
    input=None,
    stream_mode="values",):
    
    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")

In [ ]:
current_state = await client.threads.get_state(thread['thread_id'])
current_state